# LOKLOK-compatible FGD ground-truth candidate (Drive-native)

Run **after** `FGD_Multimodal_Masters_Thesis_Colab_FINAL.ipynb`. It reads the FGD package ZIP that the FGD notebook saved on your Drive (`FGD_DIR/FGD_COMPLETE_OUTPUT/FGD_THESIS_PACKAGE.zip`) and emits a structurally compatible but **review-blocked** `fgd-s2/session.json` package.

The candidate is deliberately blocked from promotion until the researcher:
- corrects the transcript against the source recording,
- replaces S0 / UNVERIFIED labels with verified moderator/participant pseudonyms,
- completes the question map,
- reviews every auto-code (especially negatives and disagreements).

In [ ]:
# Optional: prefer reading the FGD output ZIP directly from your Drive.
from google.colab import drive, files
from pathlib import Path
import zipfile, json, shutil, hashlib

FGD_DIR = Path("/content/drive/MyDrive/FGD")
ZIP_ON_DRIVE = FGD_DIR / "FGD_COMPLETE_OUTPUT" / "FGD_THESIS_PACKAGE.zip"

archive = None
if ZIP_ON_DRIVE.exists():
    archive = str(ZIP_ON_DRIVE)
    print("Using Drive ZIP:", archive)
else:
    print("FGD output ZIP not found on Drive yet. Upload it now.")
    up = files.upload()
    archive = next(iter(up))
    assert archive.lower().endswith(".zip")

SRC = Path("/content/fgd_input"); SRC.mkdir(exist_ok=True)
with zipfile.ZipFile(archive) as z:
    z.extractall(SRC)
print("Extracted to:", SRC)

# Optional: persist the source ZIP hash for provenance.
with open(archive, "rb") as fh:
    print("source_zip_sha256:", hashlib.sha256(fh.read()).hexdigest())

In [ ]:
import pandas as pd
import json, re
from pathlib import Path

df_csv = next(SRC.glob("transcription.csv"), None)
assert df_csv is not None, "transcription.csv missing from the FGD package."
df = pd.read_csv(df_csv)
assert {"text"}.issubset(df.columns), "transcription.csv must contain a 'text' column."

# Columns may be segment_id/compact_start/compact_end (final notebook) or
# segment_id/start/end (older notebooks); normalise.
for a, b in [("start", "compact_start"), ("end", "compact_end")]:
    if b not in df.columns and a in df.columns:
        df[b] = df[a].astype(float)

df["speaker"] = df.get("speaker") if "speaker" in df.columns else "S0"
df["speaker"] = df["speaker"].fillna("S0")

# Exact LOKLOK code -> anchor, direction. Arabic terms make this suitable for Arabic/English FGD text.
CB = {
  "peer_validation": ("SV",1,["friend","share","follow","like","rating","popular","audience","صديق","مشاركة","متابعة","إعجاب","تقييم","جمهور"]),
  "host_trust": ("BR",1,["trust","credible","verify","source","correct","reliable","badge","ثقة","موثوق","مصداقية","تحقق","مصادر","تصحيح","شارة"]),
  "host_mistrust": ("BR",-1,["exaggerate","clickbait","fake","distrust","false","مبالغة","مزيف","غير موثوق","كاذب"]),
  "content_quality": ("IQ",1,["informative","useful","valuable","quality","insightful","مفيد","قيم","جودة","عميق"]),
  "content_quality_neg": ("IQ",-1,["boring","shallow","repetitive","clutter","ممل","سطحي","مكرر","فوضوي"]),
  "misinfo_concern": ("CRIT",1,["misinformation","propaganda","cross-check","misleading","أخبار كاذبة","دعاية","مضلل","تحقق"]),
  "ride_positive": ("PB",1,["comfortable","safe","on-time","clean","convenience","مريحة","آمنة","نظيفة"]),
  "ride_negative": ("PB",-1,["delay","unsafe","cancel","expensive","traffic","تأخر","غير آمن","إلغاء","غالي","زحمة"]),
  "willingness": ("ENG",1,["would use","interested","excited","join","subscribe","try","سأستخدم","مهتم","متحمس","سأشترك","سأجرب"]),
  "willingness_neg": ("ENG",-1,["would not","skip","delete","leave","لن أستخدم","سأغادر","سأحذف"]),
  "audio_use": ("LIST",1,["radio","podcast","listen","audio","music","راديو","بودكاست","استماع","موسيقى"]),
  "commute": ("MOB",1,["uber","careem","taxi","commute","drive","ride","أوبر","كريم","تاكسي","تنقل","قيادة"]),
  "sm_use": ("SMU",1,["instagram","tiktok","facebook","whatsapp","scroll","social media","انستجرام","تيك توك","فيسبوك","واتساب","سكرول"])
}

def hit(t):
    low = str(t).lower()
    return [(c, a, d) for c, (a, d, w) in CB.items() if any(x in low for x in w)]

df["question"] = ""
df["codes"] = df["text"].map(lambda t: [x[0] for x in hit(t)])
df["researcher_verified"] = False
df["researcher_note"] = ""
df.to_csv(SRC / "researcher_annotation_sheet.csv", index=False, encoding="utf-8-sig")

q = df[df["text"].astype(str).str.contains(r"[?؟]", regex=True)][["segment_id", "text"]].rename(columns={"text": "proposed_question"})
q.to_csv(SRC / "machine_proposed_questions.csv", index=False, encoding="utf-8-sig")

lines = ["# FGD candidate transcript — review required", "", "# Q labels and S labels must be validated by the researcher before promotion.", ""]
for r in df.itertuples():
    lines.append(f"{r.speaker}: {r.text}")
raw = "\n".join(lines) + "\n"
(SRC / "fgd-s2-source-review-required.md").write_text(raw, encoding="utf-8")

In [ ]:
# Build the review-blocked fgd-s2 candidate and save it to your Drive.
import json
from pathlib import Path

pos, neg, quotes, total = {}, {}, {}, 0
segs = []
for r in df.itertuples():
    h = hit(r.text)
    segs.append({
        "speaker": str(r.speaker),
        "question": str(getattr(r, "question", "") or ""),
        "text": str(r.text),
        "codes": [x[0] for x in h],
        "compact_start": float(r.compact_start),
        "compact_end": float(r.compact_end),
        "researcher_verified": False,
    })
    for c, a, d in h:
        total += 1
        (pos if d > 0 else neg)[a] = (pos if d > 0 else neg).get(a, 0) + 1
        quotes.setdefault(a, []).append(str(r.text))

anchors = {}
for a in sorted(set(pos) | set(neg)):
    p, n = pos.get(a, 0), neg.get(a, 0)
    h = p + n
    b = (p - n) / max(1, h)
    anchors[a] = {
        "positive": p, "negative": n, "hits": h,
        "salience": round(h / max(1, total), 4),
        "direction": round(b, 4),
        "value": round((0.5 * (1 + h) + (0.5 + 0.5 * b) * h) / (2 * (1 + h)), 4),
        "ciHalfWidth": round(1 / max(2, h ** .5 * 2), 4),
        "exemplars": quotes[a][:3],
    }

session = {
    "session": "fgd-s2",
    "source": "fgd-s2/fgd-s2-source-review-required.md",
    "status": "review_required",
    "mode": "qualitative",
    "provenance": "primary",
    "codedAt": "machine_assisted_pending_researcher_validation",
    "nSpeakers": 0,
    "nSegments": len(segs),
    "segments": segs,
    "anchors": anchors,
    "codeHits": total,
    "provenanceLabel": "quantification of a qualitative primary source (machine-assisted coded transcript; pending researcher validation)",
    "wireframeMentions": [],
    "governance": {
        "promotion_blocked": True,
        "reason": "Transcript, speaker attribution, Q&A map and codes have not yet been researcher-verified."
    },
}

OUT = FGD_DIR / "LOKLOK_FGD_CANDIDATE_FOR_REVIEW" / "fgd-s2"
OUT.mkdir(parents=True, exist_ok=True)
(OUT / "session.json").write_text(json.dumps(session, ensure_ascii=False, indent=2), encoding="utf-8")

for n in ["fgd-s2-source-review-required.md", "researcher_annotation_sheet.csv",
          "machine_proposed_questions.csv", "transcript_timestamped.txt",
          "visual_evidence_log.json", "audio_voice_technical_notes.json",
          "FGD_comprehensive_thesis_report.md"]:
    p = next(iter(SRC.glob(n)), None)
    if p:
        shutil.copy(p, OUT / n)

prov = {
    "schema_target": "LOKLOK project/data/ground_truth/qual/fgd-s2/session.json",
    "source_zip_sha256": hashlib.sha256(open(archive, "rb").read()).hexdigest(),
    "candidate_status": "review_required",
    "anchor_set": ["BR", "SV", "IQ", "CRIT", "PB", "ENG", "LIST", "MOB", "SMU"],
    "archive_rule": "Do not modify preserved LLM_AGENT_HANDOFF or SUBMISSION archives. Promote only from a reviewed working copy."
}
(OUT / "provenance_and_compatibility.json").write_text(json.dumps(prov, indent=2), encoding="utf-8")

check = """# Researcher validation and promotion checklist
- [ ] Correct transcript against original MP4/audio.
- [ ] Replace S0 / UNVERIFIED labels with verified moderator/participant pseudonyms (M, S1...).
- [ ] Complete Q labels and approve Q&A boundaries.
- [ ] Review every auto-code, especially negatives and disagreements.
- [ ] Keep evidence timestamps and a decision log.
- [ ] Confirm consent, anonymisation, and quotation permissions.
- [ ] In a separate working copy only: after all checks, set session status to coded and update nSpeakers.
- [ ] Run the project qualitative checks before any ground-truth promotion.
"""
(OUT / "RESEARCHER_VALIDATION_CHECKLIST.md").write_text(check, encoding="utf-8")

out_zip = FGD_DIR / "LOKLOK_FGD_CANDIDATE_FOR_REVIEW"
shutil.make_archive(str(out_zip / "LOKLOK_FGD_CANDIDATE_FOR_REVIEW"), "zip", out_zip)

print("READY:", out_zip)
print(*sorted(p.name for p in OUT.iterdir()), sep="\n")
print("Candidate ZIP saved on Drive:", out_zip / "LOKLOK_FGD_CANDIDATE_FOR_REVIEW.zip")

try:
    from google.colab import files
    files.download(str(out_zip / "LOKLOK_FGD_CANDIDATE_FOR_REVIEW.zip"))
except Exception:
    print("Download prompt skipped; the candidate is already on your Drive.")